# TRACER No-Segmentation Workflow

This notebook walks through the TRACER workflow when **no input cell segmentation is available** — every transcript starts unassigned (`cell_id == "-1"`). This is the canonical mode for Visium HD data (which has no nucleus segmentation), or for Xenium / similar transcript-resolved data when the segmentation is bad enough that you'd rather discover cells from gene-coherence + spatial structure than refine a noisy prior.

## How this differs from the segmented workflow

The [`segmented_workflow.ipynb`](segmented_workflow.ipynb) notebook covers the case where input `cell_id` exists and TRACER refines it. Without a segmentation prior:

- **Skip Prune**: there are no input cells whose gene-coherence to evaluate.
- **Skip Split**: there are no entities to spatially fragment.
- **Skip Initial Rescue**: there are no existing entities to absorb transcripts into.
- **All clustering work happens in Group**: every transcript starts in the unassigned pool, and Group's bin-based clustering produces the candidate entities from scratch.

Net pipeline:

| step | name | what it does |
|---|---|---|
| 0 | Bootstrap NPMI | compute a panel-wide gene-pair PMI matrix (one-time) |
| 1 | **Group** | bin all transcripts at G=8 µm; each bin becomes one candidate component |
| 2 | **Stitch** | merge gene-compatible neighboring components |
| 3 | **Demote** | drop components below the size threshold |
| 4 | **Final Rescue** | absorb residual unassigned transcripts into surviving entities |

All output entities are **components** (`UNASSIGNED_<i>`). There are no "cells" or "partials" because no input segmentation existed to inherit those labels from.

For the broader algorithmic diff between this branch's workflow and the legacy pipeline (graph primitives, rescue rules, NPMI panel, output column lifecycle, etc.), see [`algorithmic_changes.ipynb`](algorithmic_changes.ipynb).

## Setup

In [ ]:
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

from tracer.data import discover_data_files, load_full_df
from tracer.metrics import compute_npmi_bootstrap
from tracer.pruning import prune_transcripts_fast, prune_genes_by_npmi_greedy
from tracer.spatial import (
    annotate_unassigned_components_fast,
    reassign_unassigned_grid_pool,
    demote_small_entities,
)
from tracer.stitching import apply_stitching_to_transcripts_memory_efficient
from tracer.graph import build_grid_graph_xy

### Project folder convention

The package expects a project folder laid out like:

```
<project>/
  data/
    <name>.parquet         # input transcript table
    npmi_bs*.csv           # bootstrap PMI cache (optional)
```

We use `tracer.data.discover_data_files` to find both files. The transcript table is loaded **as-is** — under no-segmentation we'll then strip the `cell_id` column to simulate the no-segmentation scenario.

In [ ]:
PROJECT_DIR = Path("/Users/adeshpa6/1_Projects/01.10_Lab/GENESIS/tutorials/lung_cancer")

parquet_path, npmi_cache_path = discover_data_files(PROJECT_DIR)
print(f"data parquet:  {parquet_path}")
print(f"NPMI cache:    {npmi_cache_path or '(none — will compute)'}")

df_input = load_full_df(project_dir=PROJECT_DIR)
print(f"\nLoaded {len(df_input):,} transcripts.")

### Strip segmentation (simulate no-seg input)

If your input data legitimately has no `cell_id` column, this step is a no-op. For Xenium data with segmentation, we set `cell_id = "-1"` everywhere and use a per-bin `bin_id` column as the **PMI group_key** for the bootstrap NPMI panel — the gene-pair statistics are computed per-bin instead of per-cell, since cells aren't available.

In [ ]:
BIN_SIZE = 2.0  # µm — Visium HD-style 2 µm bin pitch

df = df_input.copy()
if "cell_id" in df.columns and (df["cell_id"].astype(str) != "-1").any():
    print("Stripping segmentation (cell_id → '-1') and adding bin_id column.")
    if isinstance(df["cell_id"].dtype, pd.CategoricalDtype):
        if "-1" not in df["cell_id"].cat.categories:
            df["cell_id"] = df["cell_id"].cat.add_categories(["-1"])
    df["cell_id"] = "-1"

# Add bin_id (used as PMI group_key when no segmentation exists).
bx = np.floor(df["x"].to_numpy() / BIN_SIZE).astype(np.int64)
by = np.floor(df["y"].to_numpy() / BIN_SIZE).astype(np.int64)
df["bin_id"] = pd.Series(
    [f"BIN_{a}_{b}" for a, b in zip(bx.tolist(), by.tolist())],
    index=df.index, dtype=object,
)

print(f"Input cell_id distinct: {df['cell_id'].astype(str).nunique()} (expect 1: '-1')")
print(f"Input bin_id distinct:  {df['bin_id'].nunique():,}")

### Helpers: per-stage progression tracking

In [ ]:
def classify(label: str) -> str:
    s = str(label)
    if s in ("DROP", "-1", "nan", "UNASSIGNED"):
        return "unassigned"
    if s.startswith("UNASSIGNED_"):
        return "component"
    if "-" in s:
        return "partial"
    return "cell"


def state(df: pd.DataFrame, col: str) -> dict:
    s = df[col].astype(str)
    types = s.map(classify)
    n_ent = s.groupby(types).nunique().to_dict()
    n_tx = types.value_counts().to_dict()
    return {
        "n_cells": int(n_ent.get("cell", 0)),
        "n_partials": int(n_ent.get("partial", 0)),
        "n_components": int(n_ent.get("component", 0)),
        "tx_cells": int(n_tx.get("cell", 0)),
        "tx_partials": int(n_tx.get("partial", 0)),
        "tx_components": int(n_tx.get("component", 0)),
        "tx_unassigned": int(n_tx.get("unassigned", 0)),
    }


def fmt(s: dict) -> str:
    return (
        f"cells={s['n_cells']:,}/{s['tx_cells']:,}tx, "
        f"partials={s['n_partials']:,}/{s['tx_partials']:,}tx, "
        f"components={s['n_components']:,}/{s['tx_components']:,}tx, "
        f"unassigned={s['tx_unassigned']:,}tx"
    )


stages_log: list[tuple[str, dict]] = []
_input_state = state(df.assign(_lbl=df["cell_id"].astype(str)), "_lbl")
stages_log.append(("input (cell_id all '-1')", _input_state))
print(f"Input state: {fmt(_input_state)}")

## Step 0 — Bootstrap NPMI panel

Even without segmentation, we still need a panel-wide gene-pair PMI matrix to gate the gene-coherence judgments downstream (Group's gene prune, Stitch's merge ΔC, Final Rescue's mean-PMI veto). We use **bin_id** as the PMI `group_key` since `cell_id` is constant under no-seg:

- prune positive-evidence threshold: `ln(1.5) ≈ +0.405` (50 % above independence)
- rescue negative threshold: `ln(1/3) ≈ −1.099` (3× below independence; lenient veto)

In [ ]:
PMI_THR = math.log(1.5)             # +0.405
RESCUE_NEG_THR = math.log(1.0 / 3)   # −1.099
ANNOTATE_NEG_THR = -0.1 * (PMI_THR / 0.05)  # PMI-scaled −0.1 NPMI ≡ −0.811 PMI

if npmi_cache_path is not None:
    npmi_panel = pd.read_csv(npmi_cache_path)
    print(f"Loaded cached panel from {npmi_cache_path.name}: {len(npmi_panel):,} explicit pairs")
else:
    print("Computing bootstrap NPMI panel (one-time, group_key=bin_id)...")
    t0 = time.time()
    bs = compute_npmi_bootstrap(
        df, group_key="bin_id", feature_col="feature_name",
        metric="pmi", min_expected_cooccur_for_evidence=10.0,
        seed=0, show_progress=False,
    )
    import scipy.sparse as sp
    W = bs.W_sparse if sp.isspmatrix_coo(bs.W_sparse) else bs.W_sparse.tocoo()
    genes = np.asarray(bs.genes, dtype=str)
    npmi_panel = pd.DataFrame({
        "gene_i": genes[W.row],
        "gene_j": genes[W.col],
        "NPMI":   np.asarray(W.data, dtype=np.float64),
    })
    cache_out = PROJECT_DIR / "data" / "npmi_bs_full_pmi_noseg.csv"
    npmi_panel.to_csv(cache_out, index=False)
    print(f"  Computed in {time.time()-t0:.1f}s; saved to {cache_out}")

print(f"  W value range: [{npmi_panel['NPMI'].min():.3f}, {npmi_panel['NPMI'].max():.3f}]")

### Initialise tracer_id + aux

Downstream stages expect the canonical pipeline column `tracer_id` and an `aux` dict (gene-to-index map and dense PMI matrix). We get both by calling `prune_transcripts_fast` with a permissive threshold — it has nothing to prune since `cell_id` is all `"-1"`, but it still constructs `aux`. Then we explicitly reset `tracer_id` to `"-1"` to prevent any side-effects from leaking through.

In [ ]:
_, aux = prune_transcripts_fast(
    df, npmi_panel,
    cell_id_col="cell_id", gene_col="feature_name",
    threshold=-1e9,           # no pair meets this → no prune work happens
    unassigned_id="-1",
    n_jobs=-1, show_progress=False,
)
df["tracer_id"] = "-1"
print(f"aux ready (gene_to_idx: {len(aux['gene_to_idx'])} genes; W shape: {aux['W'].shape})")

## Step 1 — Group

All transcripts are unassigned. Group bins them at G=8 µm × 8 µm and forms one component per bin (the **"one bin = one candidate cell"** semantic). No cross-bin merging at this stage — Stitch will handle that.

Within each component, a greedy NPMI prune drops genes whose pairwise PMI is below the negative threshold (`−ln(1.5)·2 ≈ −0.811`).

In [ ]:
def grid_self_graph(df_in, *, k=None, dist_threshold=None, coord_cols=("x", "y", "z")):
    return build_grid_graph_xy(
        df_in, k=k, dist_threshold=dist_threshold or 1.5, coord_cols=coord_cols,
        G=8.0, neighborhood="self", exact_distance_filter=False,
    )

t0 = time.time()
df_grouped = annotate_unassigned_components_fast(
    df_pruned=df, aux=aux,
    build_graph_fn=grid_self_graph,
    prune_fn=prune_genes_by_npmi_greedy,
    coord_cols=("x", "y", "z"),
    k=8, dist_threshold=1.5,
    min_comp_size=1,
    npmi_threshold=ANNOTATE_NEG_THR,
    entity_col="tracer_id",
    out_col="tracer_id",
    cell_id_col="cell_id", gene_col="feature_name",
    transcript_id_col="transcript_id",
    show_progress=False,
)
s = state(df_grouped, "tracer_id")
stages_log.append(("after Group", s))
print(f"Group ({time.time()-t0:.1f}s): {fmt(s)}")

## Step 2 — Stitch

Group produces ~one component per occupied 8 µm bin — many of them are sub-cell fragments that should be merged. Stitch enumerates candidate merge pairs at G=2 µm 8-conn, computes per-merge ΔC (count-based coherence change at the PMI threshold), and accepts merges with ΔC ≥ 0 and `simplicity-penalized` score ≥ threshold.

Under no-seg, Stitch is the primary consolidation step — Group produces ~10× more components than the final partition will have.

In [ ]:
df_grouped["post_stage4"] = df_grouped["tracer_id"]

t0 = time.time()
df_stitched, _ = apply_stitching_to_transcripts_memory_efficient(
    df_final=df_grouped, aux=aux,
    entity_col="post_stage4", gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    mode="count", threshold=PMI_THR, metric="pmi",
    penalize_simplicity=True, deltaC_min=0.0,
    dist_threshold=5.0,
    out_col="stitched",
    show_progress=False,
    candidate_source="grid", G=2.0, stitch_neighborhood="8",
)
s = state(df_stitched, "stitched")
stages_log.append(("after Stitch", s))
print(f"Stitch ({time.time()-t0:.1f}s): {fmt(s)}")

## Step 3 — Demote

Drop entities with fewer than 5 transcripts. Their transcripts go back to `"-1"` and become eligible for Final Rescue.

In [ ]:
t0 = time.time()
df_stitched, n_demoted = demote_small_entities(
    df_stitched, entity_col="stitched", out_col="stitched",
    min_size=5, unassigned_label="-1",
)
s = state(df_stitched, "stitched")
stages_log.append(("after Demote", s))
print(f"Demote ({time.time()-t0:.1f}s): n_demoted={n_demoted:,} tx")
print(f"  {fmt(s)}")

## Step 4 — Final Rescue

Some transcripts are still unassigned: bin-cliques that didn't reach 5 tx after Stitch, or lone transcripts in sparse regions. Final Rescue tries to absorb them into surviving components using the **distance-priority + mean-PMI veto** rule:

1. **Spatial gating**: candidate entities are those whose member transcripts are in the 9-cell xy Moore neighborhood at G=2 µm, within `|Δz| ≤ G·√2 ≈ 2.83 µm`.
2. **Mean-PMI veto**: an entity is rejected if the mean PMI between the candidate gene and the entity's gene panel is `≤ 0`.
3. **Distance ranking**: pick the surviving entity with the smallest min-tx distance.

In [ ]:
t0 = time.time()
df_stitched, n_reassigned, post_rescue_stats = reassign_unassigned_grid_pool(
    df_stitched, aux=aux,
    entity_col="stitched", gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    out_col="stitched",
    G=2.0,
    pos_npmi_threshold=PMI_THR,
    neg_npmi_threshold=RESCUE_NEG_THR,
    only_partial_component=False,
    veto_mode="mean",
    mean_threshold=0.0,
    small_entity_guard_n=0,
)
s = state(df_stitched, "stitched")
stages_log.append(("after Final Rescue (FINAL)", s))
print(f"Final Rescue ({time.time()-t0:.1f}s): rescued={n_reassigned:,} tx")
print(f"  {fmt(s)}")

## Stage progression table

How counts evolve through each stage. Under no-seg there are no "cells" or "partials" by construction — every entity is a component.

In [ ]:
rows = []
for name, s in stages_log:
    n_total = s["tx_cells"] + s["tx_partials"] + s["tx_components"] + s["tx_unassigned"]
    n_assigned = s["tx_cells"] + s["tx_partials"] + s["tx_components"]
    pct = 100 * n_assigned / max(n_total, 1)
    rows.append({
        "stage": name,
        "n_components": s["n_components"],
        "tx_components": s["tx_components"],
        "tx_unassigned": s["tx_unassigned"],
        "pct_assigned": round(pct, 2),
    })
progression = pd.DataFrame(rows)
print(progression.to_string(index=False))

### Quick QC: per-component transcript-count quartiles

In [ ]:
sizes = df_stitched.loc[df_stitched["stitched"].astype(str) != "-1", "stitched"].astype(str).value_counts()
if len(sizes) > 0:
    q = np.quantile(sizes, [0.25, 0.5, 0.75, 0.9])
    print(f"components: n={len(sizes):,}")
    print(f"  size quartiles: Q1={int(q[0])}, med={int(q[1])}, Q3={int(q[2])}, p90={int(q[3])}, max={int(sizes.max())}")
    print(f"  total tx in components: {sizes.sum():,}")

---

## Summary

The no-segmentation workflow takes raw transcripts and applies four transformations:

1. **Group** bins all transcripts at 8 µm and forms one candidate component per bin.
2. **Stitch** consolidates gene-compatible adjacent components — this is the main clustering step.
3. **Demote** drops sub-5-tx components.
4. **Final Rescue** absorbs residual unassigned transcripts into the surviving components.

All output entities are `UNASSIGNED_<i>` components — there are no "cells" or "partials" because no input segmentation existed.

### Notes on Stage 4 (Split)

The segmented workflow runs a Split stage (3D bin-CC) right after Prune to break Mickey-Mouse-shape input cells. Under no-seg, Split is **omitted** because:

- There are no input cells that could be Mickey-Mouse-shaped.
- Group's same-bin components are spatially bounded to one 8 µm × 8 µm bin by construction (xy diameter ≤ 8√2 ≈ 11.3 µm). They can't be "spatially disconnected" at the Stage 4 reach (~2.83-5 µm).
- Empirically, running Split between Group and Stitch on no-seg data fragments same-bin components into smaller pieces that Demote then drops, hurting coverage by ~2-2.5pp without quality benefit.

If you have a segmented input, see the companion `segmented_workflow.ipynb` notebook.